# Silver — Central Bank Indicators
Spark SQL. Parses dates, maps the Central Bank series to metric names and keeps the latest version of each observation.

**Source:** `bronze.central_bank_raw`  
**Output:** `silver.central_bank_indicators` (Delta), one row per date and metric  
**Metrics:** `policy_rate`, `cpi`, `iskeur`, `iskusd`


In [ ]:
-- Target table (created on the first run)
CREATE TABLE IF NOT EXISTS silver.central_bank_indicators (
    date          DATE,
    metric        STRING,
    value         DOUBLE,
    processed_at  TIMESTAMP
) USING DELTA

In [ ]:
-- Upsert: the latest ingested version of each (date, metric)
MERGE INTO silver.central_bank_indicators AS target
USING (
    WITH transformed AS (
        SELECT
            TO_DATE(date_raw, 'M/d/yyyy h:mm:ss a') AS date,
            CAST(value_raw AS DOUBLE)               AS value,
            CASE
                WHEN series_name = 'Meginvextir (vextir á 7 daga bundnum innlánum)' THEN 'policy_rate'
                WHEN series_name = 'Vísitala neysluverðs' THEN 'cpi'
                WHEN series_name = 'Evra' THEN 'iskeur'
                WHEN series_name = 'Bandaríkjadalur' THEN 'iskusd'
            END AS metric,
            ingested_at
        FROM bronze.central_bank_raw
    ),
    latest AS (
        SELECT date, metric, value,
               ROW_NUMBER() OVER (PARTITION BY date, metric ORDER BY ingested_at DESC) AS rn
        FROM transformed
        WHERE date IS NOT NULL AND metric IS NOT NULL AND value IS NOT NULL
    )
    SELECT date, metric, value, CURRENT_TIMESTAMP() AS processed_at
    FROM latest
    WHERE rn = 1
) AS source
ON target.date = source.date AND target.metric = source.metric
WHEN MATCHED AND target.value <> source.value THEN UPDATE SET
    target.value        = source.value,
    target.processed_at = source.processed_at
WHEN NOT MATCHED THEN INSERT (date, metric, value, processed_at)
    VALUES (source.date, source.metric, source.value, source.processed_at)

In [ ]:
-- Check
SELECT metric, COUNT(*) AS observations, MIN(date) AS first_date, MAX(date) AS last_date
FROM silver.central_bank_indicators
GROUP BY metric
ORDER BY metric